# 00b_test_prep — Test에 Stage 1 적용 (담당: core만)

- **사전 고정(`99_freeze`) 완료 뒤 한 번만** 실행합니다. `DRY_RUN = False`에서는 `STAGE = "final"`(미결 0개)과 함께 **`verify_freeze()`가 `FREEZE_RECORD.json`의 코드·계약·데이터·모형·선택 기록 일치를 통과해야만** Test 원본을 엽니다.
- Test 원본은 `test_hold/lending_club_2020_test_new.csv` (없으면 `raw/lending_club_2020_test_new.csv`)를 사용합니다.
- 결과는 `data/`가 아니라 `test_hold/`에 저장하고(`test_prep_meta.json` 포함), eval 담당에게 직접 넘깁니다. 팀 Drive의 `data/`에는 올리지 않습니다.
- Dev와 **같은 함수, 같은 표본 규칙**(`SAMPLE`)을 씁니다. 범주는 Dev에서 고정한 `contract/categories.json`을 그대로 씁니다.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
import datetime

DRY_RUN = False   # True: Test 대신 Dev CSV로 코드만 시험 (출력은 test_hold/_dryrun_*)
FREEZE_INFO = None
if not DRY_RUN:
    assert STAGE == "final" and not PENDING, "사전 고정 커밋(STAGE='final', 미결 0개) 뒤에만 실행합니다."
    FREEZE_INFO = verify_freeze(require_final=True)
    print(f"[PASS] 사전 고정본 확인: {FREEZE_INFO['freeze_dir'].name} (sha256 {FREEZE_INFO['freeze_record_sha256'][:12]})")
TEST_DIR = ROOT / "test_hold"
TEST_DIR.mkdir(parents=True, exist_ok=True)
_test_cand = TEST_DIR / "lending_club_2020_test_new.csv"
TEST_CSV = TRAIN_CSV if DRY_RUN else (_test_cand if _test_cand.exists() else RAW_DIR / "lending_club_2020_test_new.csv")
PREFIX = "_dryrun_" if DRY_RUN else ""
print(config_fingerprint("shared"), LIB_PREP_HASH, TEST_CSV.name)

In [ ]:
# 1. Stage 1 (Dev와 같은 함수와 표본 규칙)
import time
t = time.time()
raw_X_t, meta_t = load_and_prepare(TEST_CSV, TREASURY_CSV, is_labeled=True, **SAMPLE)
print(f"{time.time() - t:.0f}초 | raw_X {raw_X_t.shape} | meta {meta_t.shape}")

In [ ]:
# 2. 검사
dev_X = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet")
dev_meta = pd.read_parquet(DATA_DIR / "dev_meta.parquet")
cats = load_categories()
issue = pd.to_datetime(meta_t["issue_d"], format="%b-%Y")
checks = {
    "열 이름과 순서가 Dev와 같음": list(raw_X_t.columns) == list(dev_X.columns),
    "X와 meta 행 수 같음": len(raw_X_t) == len(meta_t),
    "y는 0/1만": set(meta_t["y"].unique()) <= {0, 1},
    "국채 매칭 실패 없음": meta_t["rf_ann"].notna().all(),
    "MIRR 결측·무한대 없음": np.isfinite(meta_t["mirr"]).all(),
    "금지 칼럼이 X에 없음": not (FORBIDDEN_COLS & set(raw_X_t.columns)),
    "행 번호 0..N-1": raw_X_t.index.equals(pd.RangeIndex(len(raw_X_t))),
    f"발행 {SAMPLE['start_year']}-01 이후": issue.min() >= pd.Timestamp(f"{SAMPLE['start_year']}-01-01"),
}
if not DRY_RUN:
    checks["Dev와 id 겹침 없음"] = not set(meta_t["id"]) & set(dev_meta["id"])
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
print(f"발행 기간 {issue.min():%Y-%m} ~ {issue.max():%Y-%m}")
for c, levels in cats.items():
    unseen = (~raw_X_t[c].isin(levels)).mean()
    print(f"{c}: Dev 범주 밖 비율 {unseen:.2%} (모형 입력 때 OTHER로 처리)")
assert all(checks.values()), "FAIL 항목이 있음"

In [ ]:
# 3. 저장 (test_hold/에만) + 무결성 메타(test_prep_meta.json)
TEST_DIR.mkdir(parents=True, exist_ok=True)
x_out = TEST_DIR / f"{PREFIX}test_X_raw.parquet"
m_out = TEST_DIR / f"{PREFIX}test_meta.parquet"
raw_X_t.to_parquet(x_out, index=False)
meta_t.to_parquet(m_out, index=False)

test_meta_rec = dict(
    created=datetime.datetime.now().isoformat(timespec="seconds"),
    dry_run=DRY_RUN,
    n_rows=int(len(meta_t)),
    test_csv=TEST_CSV.name,
    test_csv_sha256=sha256_file(TEST_CSV),
    test_files={x_out.name: sha256_file(x_out), m_out.name: sha256_file(m_out)},
    dev_files={n: sha256_file(DATA_DIR / n) for n in ("dev_X_raw.parquet", "dev_meta.parquet")},
    freeze_dir=(FREEZE_INFO["freeze_dir"].name if FREEZE_INFO else None),
    freeze_record_sha256=(FREEZE_INFO["freeze_record_sha256"] if FREEZE_INFO else None),
    sample=SAMPLE,
    config_hash=config_fingerprint("shared"),
    fingerprints=config_fingerprints(),
    lib_prep_hash=LIB_PREP_HASH,
    meta=run_meta(),
)
(TEST_DIR / f"{PREFIX}test_prep_meta.json").write_text(
    json.dumps(test_meta_rec, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8"
)
print("저장:", sorted(p.name for p in TEST_DIR.glob(f"{PREFIX}test_*")))
print(f"{len(meta_t):,}행. eval 담당에게 test_hold/의 파일(parquet 2개 + {PREFIX}test_prep_meta.json)을 직접 넘깁니다.")